# Decision Tree (CART vs ID3) - Automated Pipeline

This notebook loads dataset from GitHub RAW link and compares CART (Gini) vs ID3 (Entropy).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA


In [ ]:
# =========================
# LOAD DATA (GITHUB RAW)
# =========================

DATA_URL = 'https://raw.githubusercontent.com/USERNAME/REPO/main/salary_dataset.csv'

df = pd.read_csv(DATA_URL)
df.head()

In [ ]:
# =========================
# PREPROCESSING
# =========================

df = df.dropna()

# encode categorical columns
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# assume last column is target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# =========================
# CART MODEL (GINI)
# =========================

cart = DecisionTreeClassifier(criterion='gini')

params = {
    'max_depth': [3,5,10,None],
    'min_samples_split': [2,5,10]
}

grid_cart = GridSearchCV(cart, params, cv=5)
grid_cart.fit(X_train, y_train)

best_cart = grid_cart.best_estimator_
best_cart

In [ ]:
# =========================
# ID3 MODEL (ENTROPY)
# =========================

id3 = DecisionTreeClassifier(criterion='entropy')

grid_id3 = GridSearchCV(id3, params, cv=5)
grid_id3.fit(X_train, y_train)

best_id3 = grid_id3.best_estimator_
best_id3

In [ ]:
# =========================
# PREDICTIONS
# =========================

cart_pred = best_cart.predict(X_test)
id3_pred = best_id3.predict(X_test)

In [ ]:
# =========================
# METRICS FUNCTION
# =========================

def get_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall': recall_score(y_true, y_pred, average='weighted'),
        'f1': f1_score(y_true, y_pred, average='weighted')
    }

cart_metrics = get_metrics(y_test, cart_pred)
id3_metrics = get_metrics(y_test, id3_pred)

cart_metrics, id3_metrics

In [ ]:
# =========================
# CONFUSION MATRIX (2x1)
# =========================

fig, ax = plt.subplots(1,2, figsize=(12,5))

sns.heatmap(confusion_matrix(y_test, cart_pred), annot=True, fmt='d', ax=ax[0])
ax[0].set_title('CART Confusion Matrix')

sns.heatmap(confusion_matrix(y_test, id3_pred), annot=True, fmt='d', ax=ax[1])
ax[1].set_title('ID3 Confusion Matrix')

plt.show()

In [ ]:
# =========================
# METRICS BAR CHART
# =========================

labels = ['Accuracy','Precision','Recall','F1']

cart_vals = list(cart_metrics.values())
id3_vals = list(id3_metrics.values())

x = np.arange(len(labels))

plt.figure(figsize=(8,5))
plt.bar(x-0.2, cart_vals, width=0.4, label='CART')
plt.bar(x+0.2, id3_vals, width=0.4, label='ID3')

plt.xticks(x, labels)
plt.legend()
plt.title('Model Comparison')
plt.show()

In [ ]:
# =========================
# TREE VISUALIZATION
# =========================

plt.figure(figsize=(15,8))
plot_tree(best_cart, filled=True)
plt.title('Best CART Tree')
plt.show()

## NOTE:
- Update DATA_URL with your GitHub raw CSV link
- Ensure target column is last column
